# 05 - Quantize, Export, and Inference Smoke Tests


# Runtime + Repo Setup
This notebook is designed for **Google Colab Free** first, with CPU fallback.
It installs dependencies, detects runtime, and keeps training defaults small for low-memory constraints.


In [ ]:
import os, platform, subprocess, sys, json

print('Python:', sys.version)
print('Platform:', platform.platform())
print('COLAB_GPU:', os.environ.get('COLAB_GPU', 'None'))
try:
    import torch
    print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'mps:', hasattr(torch.backends, 'mps') and torch.backends.mps.is_available())
except Exception as exc:
    print('torch not ready yet:', exc)


In [ ]:
# If repo is mounted in /content, keep these paths as-is.
!pip install -q -r requirements.txt
!pip install -q -e .


## Quantize / export all model checkpoints


In [ ]:
!PYTHONPATH=src python -m myvideo.keyframe.quantize --checkpoint models/checkpoints/keyframe_tiny.pt
!PYTHONPATH=src python -m myvideo.inbetween.quantize --checkpoint models/checkpoints/inbetween_tiny.pt
!PYTHONPATH=src python -m myvideo.upscale.quantize --checkpoint models/checkpoints/upscaler_tiny.pt
!PYTHONPATH=src python -m myvideo.interpolate.quantize --checkpoint models/checkpoints/interpolator_tiny.pt


## Create timeline from prompt and run low-memory make pipeline


In [ ]:
%%bash
cat > prompt.txt << 'TXT'
A cinematic one-minute sequence of a neon city sunrise, slow camera movement, and calm atmosphere.
TXT

PYTHONPATH=src python -m myvideo.cli.main plan --prompt-file prompt.txt --output-json outputs/timeline.json --duration 60


## Verify exported artifacts


In [ ]:
from pathlib import Path
for p in [
    'models/quantized/keyframe_tiny_int8.pt',
    'models/quantized/inbetween_tiny_fp16.pt',
    'models/quantized/upscaler_tiny_fp16.pt',
    'models/quantized/interpolator_tiny_fp16.pt',
]:
    print(p, Path(p).exists())
